In [27]:
import sys
from pathlib import Path
from dotenv import load_dotenv
# Production layout: add project root and src for imports (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
load_dotenv(_root / ".env")
load_dotenv("/app/.env")
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.postgres.news_dataframe import (
    filter_financial_news_by_date,
    filter_financial_news_ingested_today,
    filter_financial_news_published_today,
    get_financial_news_content_by_id,
    normalize_financial_news_datetime_column,
)
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import date, datetime

# Sanity check: if this fails, use File → Reload Notebook from Disk, then restart kernel
import storage.postgres.news_dataframe as _news_df
print(f"Using news_dataframe from: {_news_df.__file__}")
print(f"filter_financial_news_ingested_today: OK")

Using news_dataframe from: /app/src/storage/postgres/news_dataframe.py
filter_financial_news_ingested_today: OK


In [28]:
table_name = PostgresSQL_table_queries.FINANCIAL_NEWS_TABLE_NAME
pg_conn = PgConn(table_name)
df = pg_conn.get_financial_news()
if df is None:
    raise RuntimeError(
        "get_financial_news() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: financial_news_241118


In [29]:
print(f"Total news articles queried: {df.shape[0]}")

Total news articles queried: 98


In [30]:
df.head()

,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,3576811985894112344,BeInCrypto,$40 Trillion US Debt: Could Americans Even Aff...,https://finance.yahoo.com/markets/crypto/artic...,,The $40 trillion US debt mark is now within re...,Lockridge Okoth,4 min read,2026-08-18 22:24:59,2026-08-19 14:36:30.906755
1,1566228674119227985,MarketBeat,Antalpha Platform Q2 Earnings Call Highlights,https://finance.yahoo.com/markets/crypto/artic...,,Key Points\nInterested in Antalpha Platform Ho...,MarketBeat,6 min read,2026-08-19 13:03:18,2026-08-19 13:50:08.190852
2,1183237311810909673,BeInCrypto,Anthropic Copies Elon Musk’s SpaceX IPO Playbo...,https://finance.yahoo.com/markets/stocks/artic...,,Anthropic reportedly plans to hand CEO Dario A...,Lockridge Okoth,3 min read,2026-08-18 20:52:41,2026-08-19 14:36:30.915292
3,2424526424683673230,BeInCrypto,Arthur Hayes’ New Token Will Airdrop Before It...,https://finance.yahoo.com/markets/crypto/artic...,,AI job losses weigh on entry-level workers acr...,Lockridge Okoth,3 min read,2026-08-18 15:43:43,2026-08-19 14:36:31.030453
4,3596659142102186242,BeInCrypto,Bank of America Thinks Nvidia Stock Could Go 5...,https://finance.yahoo.com/markets/stocks/artic...,,Wall Street fears Nvidia (NVDA) is quietly tur...,Lockridge Okoth,3 min read,2026-08-18 19:29:15,2026-08-19 14:36:30.940667


In [31]:
def delete_records_for_current_date(df, pg_conn):
    if df is None:
        print("DataFrame is empty. No records to delete.")
        return
        
    today_records = filter_financial_news_by_date(df)

    # Extract date strings (DB/delete API expects stored string form)
    date_strings = today_records['datetime'].astype(str).tolist()

    # Call the delete_records_by_date method
    pg_conn.delete_records_by_date(date_strings)

# Then call the delete_records_for_current_date method
#delete_records_for_current_date(df, pg_conn)
#list_ids = ["11111"]
#pg_conn.delete_records_by_ids(list_ids)

In [32]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
    class Export():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def set_dataframe(self, dataframe):
            self.df = dataframe
        
        def export_text_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_datetime_subfolders(self.df, bucket_name, prefix_path, file_format)
        
        def export_text_to_s3_full_file(self, bucket_name, prefix_path, filename):
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
        
        def get_data_csv_file_by_datetime(
            self, bucket_name, prefix_path, year, month, day, hour=None, minute=None, second=None
        ):
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(
                bucket_name,
                prefix_path,
                year=year,
                month=month,
                day=day,
                hour=hour,
                minute=minute,
                second=second,
            )
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            # Article publish date (datetime) on today's calendar date (default on_date=None)
            return filter_financial_news_by_date(self.df)
            
    class Transform():
        def extractStopWords():
            pass

In [33]:
etl = DataETL(df)

# Export by article publish time (datetime). on_date=None → today's local date (date.today()).
export_on_date = "2026-08-19"  # e.g. "2026-05-23" to override today
filtered_df = filter_financial_news_by_date(df, on_date=export_on_date)

print(
    f"Filter date (datetime column): {export_on_date or date.today()} | "
    f"Rows matched: {len(filtered_df)} | "
    f"Ingested today (created_at only): {len(filter_financial_news_ingested_today(df))}"
)
filtered_df.head()

Filter date (datetime column): 2026-08-19 | Rows matched: 47 | Ingested today (created_at only): 98


,id,source,headline,href,summary,content,author,minsread,datetime,created_at
1,1566228674119227985,MarketBeat,Antalpha Platform Q2 Earnings Call Highlights,https://finance.yahoo.com/markets/crypto/artic...,,Key Points\nInterested in Antalpha Platform Ho...,MarketBeat,6 min read,2026-08-19 13:03:18,2026-08-19 13:50:08.190852
5,4559075629343239001,BeInCrypto,Bank of Italy Study Finds Stablecoins No Cheap...,https://finance.yahoo.com/markets/crypto/artic...,,Stablecoin remittances tested by the Bank of I...,Darryn Pollock,2 min read,2026-08-19 02:02:52,2026-08-19 14:36:30.891606
6,2297384304687520222,Zacks,Bear of the Day: Coinbase Global (COIN),https://finance.yahoo.com/markets/crypto/artic...,,Coinbase Global COIN) has spent the past sever...,Shaun Pruitt,4 min read,2026-08-19 08:00:00,2026-08-19 14:36:30.846810
7,1138155509936162494,TheStreet,Billionaire's dig at millionaires earns brickbats,https://finance.yahoo.com/markets/crypto/artic...,,"Changpeng ""CZ"" Zhao, the billionaire co-founde...",Anand Sinha,2 min read,2026-08-19 00:35:17,2026-08-19 14:36:30.897785
8,3055495202849118381,CCN,"Bitcoin Crashed 53%, Yet BlackRock Says a 1%-2...",https://finance.yahoo.com/markets/crypto/artic...,,"Bitcoin plunged 53%, but BlackRock says a 1%-2...",Giuseppe Ciccomascolo,4 min read,2026-08-19 11:13:40,2026-08-19 14:36:30.769611


In [34]:
print(f"Total filtered news articles queried: {filtered_df.shape[0]}")

Total filtered news articles queried: 47


In [35]:
import os

export_rows_to_s3 = True
etl_export = etl.Export(filtered_df)
bucket_name = "test-financial-news-bucket"
prefix_path = "news/crypto"
file_format = "csv"

if export_rows_to_s3 and not filtered_df.empty:
    if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
        raise RuntimeError(
            "AWS credentials not configured. Uncomment and set AWS_ACCESS_KEY_ID and "
            "AWS_SECRET_ACCESS_KEY in .env, then restart Jupyter: ./docker/start_jupyter.ps1"
        )
    export_df = normalize_financial_news_datetime_column(filtered_df)
    etl_export.set_dataframe(export_df)
    etl_export.export_text_to_s3(bucket_name, prefix_path, file_format)

Bucket 'test-financial-news-bucket' already exists.
Data for row 1 with id '1566228674119227985' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=19/hour=13/minute=03/second=18/format=csv/1566228674119227985.csv'
Data for row 5 with id '4559075629343239001' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=19/hour=02/minute=02/second=52/format=csv/4559075629343239001.csv'
Data for row 6 with id '2297384304687520222' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=19/hour=08/minute=00/second=00/format=csv/2297384304687520222.csv'
Data for row 7 with id '1138155509936162494' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=19/hour=00/minute=35/second=17/format=csv/1138155509936162494.csv'
Data for row 8 with id '3055495202849118381' uploaded to S3 bucket 'test-financial-news-bucket' unde

In [36]:
post_full_csv = False
if (post_full_csv == True) and not filtered_df.empty:
    now = datetime.now()
    filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
    etl_export.set_dataframe(etl.df)
    etl_export.export_text_to_s3_full_file(bucket_name, prefix_path, filename)

In [37]:
ingest_data = False
get_full_file = False
get_by_datetime = True
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-news-bucket"
    prefix_path = "news/crypto/"
    year = '2026'
    month = '05'
    day = '24'
    hour = None   # set e.g. '04' to narrow to one hour; None = whole day
    minute = None
    if get_full_file == True:
        filename = f"{year}-{month}-{day}_full_record.csv"
        full_path = f"{prefix_path}{filename}"
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, full_path)
    elif get_by_datetime == True:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(bucket_name, prefix_path, year, month, day, hour, minute)

In [38]:
if df_from_file is not None:
    print(df_from_file.count())
    df_from_file.head()

In [39]:
targetId = ""  # i.e. "1221589746717124508" targetId can be string or numeric type

# Lookup order: S3 ingest result, filtered export batch, then full DB pull
lookup_df = None
lookup_source = None
for name, candidate in (
    ("df_from_file", df_from_file if "df_from_file" in dir() else None),
    ("filtered_df", filtered_df if "filtered_df" in dir() else None),
    ("df", df if "df" in dir() else None),
):
    if candidate is not None and not getattr(candidate, "empty", True):
        lookup_df = candidate
        lookup_source = name
        break

if targetId and lookup_df is not None:
    full_content = get_financial_news_content_by_id(lookup_df, targetId)
    if full_content:
        print(full_content)
    else:
        print(
            f"No content for id {targetId!r} in {lookup_source} "
            f"({len(lookup_df)} rows). Id column dtype: {lookup_df['id'].dtype}"
        )
else:
    print("Set targetId and ensure df_from_file, filtered_df, or df is loaded.")

Set targetId and ensure df_from_file, filtered_df, or df is loaded.
